# **Chronic_Kidney_Disease**

In [1]:
pip install scipy

In [2]:
pip install liac-arff

  Preparing metadata (setup.py) ... done
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=42ded9c216e2ce06b749cdbc53f3a199ad3d75ab0ca2e7817108a2603fad7ef5
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [3]:
import pandas as pd

file_path = "/content/drive/MyDrive/data/Chronic_Kidney_Disease/Chronic_Kidney_Disease/chronic_kidney_disease_full.arff"

data = []
columns = []
data_section = False

with open(file_path, 'r') as f:
    for line in f:
        line = line.strip()

        # Get column names
        if line.lower().startswith('@attribute'):
            columns.append(line.split()[1])

        # Detect start of data
        elif line.lower().startswith('@data'):
            data_section = True

        # Read only actual data rows
        elif data_section and line != "":
            row = line.split(',')

            # Fix column mismatch
            if len(row) > len(columns):
                row = row[:len(columns)]
            elif len(row) < len(columns):
                row += [None] * (len(columns) - len(row))

            data.append(row)

# Create DataFrame
df = pd.DataFrame(data, columns=columns)

df.head()

,'age','bp','sg','al','su','rbc','pc','pcc','ba','bgr',...,'pcv','wbcc','rbcc','htn','dm','cad','appet','pe','ane','class'
0,48,80,1.020,1,0,?,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,?,normal,notpresent,notpresent,?,...,38,6000,?,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,?,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [6]:
df.columns = df.columns.str.replace("'", "").str.strip().str.lower()

In [7]:
print(df['class'].unique())

['ckd' 'notckd' 'no']


In [8]:
print(df['class'].value_counts())

class
ckd       250
notckd    149
no          1
Name: count, dtype: int64


In [9]:
df = df[df['class'].isin(['ckd', 'notckd'])]

In [11]:
import pandas as pd

df.replace('?', pd.NA, inplace=True)

/tmp/ipykernel_1195/3954345937.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.replace('?', pd.NA, inplace=True)


In [13]:
for col in df.columns:
    if df[col].dtype != 'object':
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(df[col].median(), inplace=True)

In [14]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipykernel_1195/3598930323.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_1195/3598930323.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col].fillna(df[col].mode()[0], inplace=True)


In [15]:
df.isnull().sum()

,0
age,0
bp,0
sg,0
al,0
su,0
rbc,0
pc,0
pcc,0
ba,0
bgr,0


In [18]:
# Encode categorical data
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == object:
        df[col] = le.fit_transform(df[col])

/tmp/ipykernel_1195/2024538746.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = le.fit_transform(df[col])
/tmp/ipykernel_1195/2024538746.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = le.fit_transform(df[col])
/tmp/ipykernel_1195/2024538746.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_

In [20]:
from sklearn.model_selection import train_test_split

X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# **MODEL 1: Decision Tree for Kidney Diseases**

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Report:\n", classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.9375
Confusion Matrix:
 [[50  3]
 [ 2 25]]
Report:
               precision    recall  f1-score   support

           0       0.96      0.94      0.95        53
           1       0.89      0.93      0.91        27

    accuracy                           0.94        80
   macro avg       0.93      0.93      0.93        80
weighted avg       0.94      0.94      0.94        80



# **MODEL 2: Random Forest Kidney Diseases**

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Report:\n", classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 1.0
Confusion Matrix:
 [[53  0]
 [ 0 27]]
Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        53
           1       1.00      1.00      1.00        27

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



In [23]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

,0
sc,0.179254
pcv,0.177974
sg,0.153013
al,0.089969
hemo,0.057038
sod,0.050804
htn,0.050378
dm,0.049825
rbcc,0.048852
wbcc,0.029575


# **Mushrooms Dataset**

In [24]:
import pandas as pd

df_m = pd.read_csv("/content/drive/MyDrive/data/mushrooms.csv")
df_m.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [25]:
df_m.isnull().sum()

,0
class,0
cap-shape,0
cap-surface,0
cap-color,0
bruises,0
odor,0
gill-attachment,0
gill-spacing,0
gill-size,0
gill-color,0


In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df_m.columns:
    df_m[col] = le.fit_transform(df_m[col])

In [27]:
print(df_m['class'].unique())

[1 0]


In [28]:
from sklearn.model_selection import train_test_split

X_m = df_m.drop('class', axis=1)
y_m = df_m['class']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42
)

# **Model 1 Decision Tree for Mushroom Dataset**

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dt_m = DecisionTreeClassifier(random_state=42)
dt_m.fit(X_train_m, y_train_m)

y_pred_dt_m = dt_m.predict(X_test_m)

print("DT Accuracy (Mushroom):", accuracy_score(y_test_m, y_pred_dt_m))
print(confusion_matrix(y_test_m, y_pred_dt_m))
print(classification_report(y_test_m, y_pred_dt_m))

DT Accuracy (Mushroom): 1.0
[[843   0]
 [  0 782]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       843
           1       1.00      1.00      1.00       782

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



# **Model 2 Random Forest for Mushroom Dataset**

In [30]:
from sklearn.ensemble import RandomForestClassifier

rf_m = RandomForestClassifier(n_estimators=100, random_state=42)
rf_m.fit(X_train_m, y_train_m)

y_pred_rf_m = rf_m.predict(X_test_m)

print("RF Accuracy (Mushroom):", accuracy_score(y_test_m, y_pred_rf_m))
print(confusion_matrix(y_test_m, y_pred_rf_m))
print(classification_report(y_test_m, y_pred_rf_m))

RF Accuracy (Mushroom): 1.0
[[843   0]
 [  0 782]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       843
           1       1.00      1.00      1.00       782

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625

